# 🎨 SD WebUI Forge + Pixel Art XL + LayerDiffusion + AnimateDiff 部署

**用途**：在 Google Colab 免费 T4 GPU 上部署 Stable Diffusion WebUI Forge，
用于生成 16-bit 像素画立绘（原生透明 PNG）和 24FPS 动画帧。

**使用前**：
1. 菜单 `运行时` → `更改运行时类型` → 选 **T4 GPU**
2. 按顺序运行每个 cell
3. 最后一个 cell 会输出 `https://xxx.gradio.live` 公网 URL，用于本地 API 调用

**模型清单**（全部 HuggingFace 免费直链，无需 API Key）：
- SDXL Base 1.0 (6.9GB)
- Pixel Art XL LoRA (nerijs)
- LayerDiffusion layer_xl_transparent_attn
- AnimateDiff-SDXL mm_sdxl_v10_beta
- SDXL ControlNet Canny + Lineart

## Cell 1: 检查 GPU

In [ ]:
!nvidia-smi
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB' if torch.cuda.is_available() else '')

## Cell 2: 克隆 Forge 与扩展（约 2 分钟）

In [ ]:
import os
os.chdir('/content')

# 克隆 Forge 主程序
!rm -rf stable-diffusion-webui-forge
!git clone --depth 1 https://github.com/lllyasviel/stable-diffusion-webui-forge.git
%cd /content/stable-diffusion-webui-forge

# 安装 LayerDiffusion（透明图像生成）
!git clone --depth 1 https://github.com/lllyasviel/sd-forge-layerdiffuse.git extensions/sd-forge-layerdiffuse

# 安装 AnimateDiff（图生视频动画）
!git clone --depth 1 https://github.com/continue-revolution/sd-webui-animatediff.git extensions/sd-webui-animatediff

# 安装像素画后处理扩展
!git clone --depth 1 https://github.com/microsoft/sd-webui-pixelart.git extensions/sd-webui-pixelart 2>/dev/null || echo 'pixelart ext optional'

print('✅ 扩展克隆完成')

## Cell 3: 下载模型（约 10-15 分钟，约 10GB）

In [ ]:
import os
root = '/content/stable-diffusion-webui-forge'

# 1. SDXL Base 1.0 (fp16, 6.9GB)
!wget -c -q --show-progress -O {root}/models/Stable-diffusion/sd_xl_base_1.0.safetensors \
  https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors

# 2. SDXL VAE (修复版，避免黑图)
!wget -c -q --show-progress -O {root}/models/VAE/sdxl_vae.safetensors \
  https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/diffusion_pytorch_model.safetensors

# 3. Pixel Art XL LoRA (16-bit 像素画)
!wget -c -q --show-progress -O {root}/models/Lora/pixel-art-xl.safetensors \
  https://huggingface.co/nerijs/pixel-art-xl/resolve/main/pixel-art-xl.safetensors

# 4. LayerDiffusion 透明生成模型 (SDXL)
!mkdir -p {root}/models/layer_model
!wget -c -q --show-progress -O {root}/models/layer_model/layer_xl_transparent_attn.safetensors \
  https://huggingface.co/LayerDiffusion/layerdiffusion-v1/resolve/main/layer_xl_transparent_attn.safetensors
!wget -c -q --show-progress -O {root}/models/layer_model/layer_xl_fg2ble.safetensors \
  https://huggingface.co/LayerDiffusion/layerdiffusion-v1/resolve/main/layer_xl_fg2ble.safetensors
!wget -c -q --show-progress -O {root}/models/layer_model/layer_xl_ble2fg.safetensors \
  https://huggingface.co/LayerDiffusion/layerdiffusion-v1/resolve/main/layer_xl_ble2fg.safetensors

# 5. AnimateDiff SDXL 运动模块 (1.5GB)
!mkdir -p {root}/extensions/sd-webui-animatediff/model
!wget -c -q --show-progress -O {root}/extensions/sd-webui-animatediff/model/mm_sdxl_v10_beta.ckpt \
  https://huggingface.co/guoyww/AnimateDiff/resolve/main/mm_sdxl_v10_beta.ckpt

# 6. SDXL ControlNet Canny + Lineart (角色一致性)
!mkdir -p {root}/models/ControlNet
!wget -c -q --show-progress -O {root}/models/ControlNet/diffusion_xl_canny_full.safetensors \
  https://huggingface.co/lllyasviel/ControlNet/resolve/main/diffusion_xl_canny_full.safetensors
!wget -c -q --show-progress -O {root}/models/ControlNet/diffusion_xl_lineart_full.safetensors \
  https://huggingface.co/lllyasviel/ControlNet/resolve/main/diffusion_xl_lineart_full.safetensors

print('✅ 所有模型下载完成')
!du -sh {root}/models/*

## Cell 4: 启动 WebUI（输出 gradio.live URL）

In [ ]:
%cd /content/stable-diffusion-webui-forge
# --share: 生成公网 gradio.live URL（用于本地 API 调用）
# --xformers: 显存优化
# --no-half-vae: 避免 VAE 黑图
# --enable-insecure-extension-access: 允许扩展 API
!python launch.py --share --xformers --enable-insecure-extension-access \
  --no-half-vae --theme dark --port 7860 --listen

## 📋 下一步

Cell 4 输出会包含形如 `Running on public URL: https://xxxxx.gradio.live` 的链接。

**复制该 URL**，回到本地沙箱运行：
```bash
python /workspace/tools/sd_api_client.py --url https://xxxxx.gradio.live
```
本地脚本会自动调用 Forge API 批量生成所有角色立绘和动画。

**若断连**：重新运行 Cell 4，获取新 URL，本地脚本用 `--resume` 续跑。